In [6]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.compose import ColumnTransformer


def universal_pipeline(file_path, target_column):

    df = pd.read_csv(file_path)
    print(f"Data loaded! Shape: {df.shape}")
    print(f"Columns in the dataset: {df.columns.tolist()}")

    if target_column not in df.columns:
        print(f"Error: Target column '{target_column}' not found in the dataset. Please choose one from: {df.columns.tolist()}")
        return

    X = df.drop(columns=[target_column])
    y = df[target_column]

    # Identify categorical features (all features are categorical in this dataset)
    categorical_features = X.columns.tolist()

    # Create a preprocessor for one-hot encoding categorical features
    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
        ])

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    models = {
        "Logistic Regression" : LogisticRegression(max_iter=1000),
        "Decision Tree"       : DecisionTreeClassifier(),
        "Random Forest"       : RandomForestClassifier(),
        "KNN"                 : KNeighborsClassifier(),
    }

    print("\nModel                | Accuracy")
    print("-" * 35)

    best_acc  = 0
    best_name = ""

    for name, model in models.items():

        pipe = Pipeline([
            ('preprocessor', preprocessor),
            ("scaler", StandardScaler(with_mean=False)), # StandardScaler after OneHotEncoder requires with_mean=False if sparse matrix is outputted
            ("model",  model)
        ])

        pipe.fit(X_train, y_train)
        acc = accuracy_score(y_test, pipe.predict(X_test))
        print(f"{name:20s} | {acc:.2f}")

        if acc > best_acc:
            best_acc  = acc
            best_name = name

    print(f"\nBest Model : {best_name}")
    print(f"Accuracy   : {best_acc:.2f}")


universal_pipeline("/content/CarEval.csv", target_column="class values")

Data loaded! Shape: (1728, 7)
Columns in the dataset: ['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'class values']

Model                | Accuracy
-----------------------------------
Logistic Regression  | 0.93
Decision Tree        | 0.96
Random Forest        | 0.97
KNN                  | 0.84

Best Model : Random Forest
Accuracy   : 0.97
